# Dados extraidos

1) O primeiro passo foi extrair os dados de correspondencia efetiva no segundo turno direto do portal da transparência. Essa base contém:

- Localização eleitoral: Município, Zona, Seção e Local de Votação.  
- Identificação do equipamento: Número e código de carga da urna esperada vs. efetivamente utilizada.
- Integridade/Origem: Origem dos votos do Boletim de Urna (urna física, totalização, contingência, etc.) e tipos de divergência cadastral. 

2) Segunda base extraida foi a modelourna_numerointerno que compara a faixa de números internos em cada uma com o seu modelo o ano que foi fabricada isso permite identificar entre as urnas usadas nas regiões o real modelo(ano) de cada uma. ex:
- 2009;999500;1220500
- urnas com numero internos entre 999500 - 1220500 são de 2009

### 3) Boletim de Urna segundo turno
Variável resposta da análise: contagem agregada de votos por candidato a Presidente (QT_VOTOS, NM_VOTAVEL) em cada seção.
Essa fonte de dados possui a relação de número de votos : LULA, BOLSONARO, NULO, BRANCO ou ABSTENÇÕES. 
__Por cada combinação de Município (CD_MUNICIPIO), Zona (NR_ZONA) e Seção (NR_SECAO)__

tipos de numeros de votáveis:
- 13: Lula
- 22: Bolsonaro
- 95: Branco (eleitor comparece à seção eleitoral, digita a tecla "Branco" na urna e confirma)
- 96: Nulo (eleitor comparece à seção eleitoral, digita um número inexistente (como 00 ou 99) e confirma)

Abstenções: coluna numérica QT_ABSTENÇOES representa quantidade de pessoas que não compareceram para votar no segundo turno

### Extraindo os dados de boletim do segundo turno scrapping

In [ ]:
pip install requests bs4 curl_cffi

In [6]:
import requests
from bs4 import BeautifulSoup
from curl_cffi import requests
import zipfile
import io
import os

### Função de extracao para scrapping no tse
- funciona para qualquer scraping feito no tse, todos os links seguem o mesmo padrão, contidos numa tag A com a mesma classe: "dropdown-item resource-url-analytics"
- precisamos apenas passar a url que queremos extrair
- o padrão do link especifico
- pasta onde serão salvos os arquivos

In [8]:


def extrair_dados(url, padrao_link, pasta_destino):

  resposta = requests.get(url, impersonate="chrome120")

  soup = BeautifulSoup(resposta.text, 'html.parser')

  links = []

  for tag in soup.find_all(name='a', class_="dropdown-item resource-url-analytics"):
    #pegar o link da tag
    link = tag.get('href','')

    #verficar se é um link de segundo turno
    if padrao_link in link and link.endswith('.zip'):
      links.append(link)

  #baixar csv numa pasta
  os.makedirs(pasta_destino, exist_ok=True)

  headers = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
  }

  total = len(links)
  for i, link in enumerate(links,1):
    nome_arquivo = link.split('/')[-1]
    print(f"[{i}/{total}] Baixando e extraindo: {nome_arquivo}")

    resposta = requests.get(link, impersonate="chrome120")
    resposta.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resposta.content)) as z:
      z.extractall(pasta_destino)


### 4) Perfil do Eleitorado
conjutno de variáveis de controle da composição eleitoral (idade, escolaridade e gênero por seção).

In [9]:
url_perfil_eleitorado = "https://dadosabertos.tse.jus.br/dataset/eleitorado-2022"
padrao_link = "/perfil_eleitor_secao/"


extrair_dados(url= url_perfil_eleitorado, padrao_link= padrao_link, pasta_destino="perfil_eleitorado_2022")


[1/28] Baixando e extraindo: perfil_eleitor_secao_2022_AC.zip
[2/28] Baixando e extraindo: perfil_eleitor_secao_2022_AL.zip
[3/28] Baixando e extraindo: perfil_eleitor_secao_2022_AM.zip
[4/28] Baixando e extraindo: perfil_eleitor_secao_2022_AP.zip
[5/28] Baixando e extraindo: perfil_eleitor_secao_2022_BA.zip
[6/28] Baixando e extraindo: perfil_eleitor_secao_2022_CE.zip
[7/28] Baixando e extraindo: perfil_eleitor_secao_2022_DF.zip
[8/28] Baixando e extraindo: perfil_eleitor_secao_2022_ES.zip
[9/28] Baixando e extraindo: perfil_eleitor_secao_2022_GO.zip
[10/28] Baixando e extraindo: perfil_eleitor_secao_2022_MA.zip
[11/28] Baixando e extraindo: perfil_eleitor_secao_2022_MG.zip
[12/28] Baixando e extraindo: perfil_eleitor_secao_2022_MS.zip
[13/28] Baixando e extraindo: perfil_eleitor_secao_2022_MT.zip
[14/28] Baixando e extraindo: perfil_eleitor_secao_2022_PA.zip
[15/28] Baixando e extraindo: perfil_eleitor_secao_2022_PB.zip
[16/28] Baixando e extraindo: perfil_eleitor_secao_2022_PE.zip
[

# Integrando os dados

## 1) Unindo dado central com ano da urna
para essa infromação vamos juntar os dados de boletim do segundo turno com o mapeamento de intervalo das urnas e o modelo 
- arquivos necessários: boletim_urna_segundo_turno, modelourna_numerointerno


In [ ]:
pip install pandas

In [3]:
import pandas as pd
import os
import glob
import numpy as np

### 1.5) Unir os csv de boletim_segundo_turno em um unico dataframe

In [11]:
boletim_urna_pasta = "boletim_urna_segundo_turno"
arquivos_boletim_urna = sorted(glob.glob(os.path.join(boletim_urna_pasta, "*.csv")))

colunas_necessarias = [
    'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO',
    'DS_CARGO_PERGUNTA', 'NM_VOTAVEL', 'QT_VOTOS',
    'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES', 'NR_URNA_EFETIVADA'
]

chaves_secao = [
    'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO',
    'NR_URNA_EFETIVADA', 'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES'
]

dfs_processados = []

print(f"Iniciando consolidação de {len(arquivos_boletim_urna)} arquivos de Boletim de Urna...")

for i, arquivo in enumerate(arquivos_boletim_urna, 1):
    nome_arq = os.path.basename(arquivo)
    
    # Leitura apenas das colunas necessárias para economizar RAM
    df_temp = pd.read_csv(
        arquivo,
        sep=';',
        encoding='latin1',
        usecols=colunas_necessarias,
        dtype=str,
        on_bad_lines='skip'
    )
    
    # Filtra Presidente no 2º turno
    df_pres = df_temp[df_temp['DS_CARGO_PERGUNTA'] == 'Presidente'].copy()
    if df_pres.empty:
        continue
        
    df_pres['QT_VOTOS'] = pd.to_numeric(df_pres['QT_VOTOS'], errors='coerce').fillna(0)
    
    # Transforma as linhas de candidatos em colunas únicas por seção
    df_secao = df_pres.pivot_table(
        index=chaves_secao,
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        fill_value=0
    ).reset_index()
    
    df_secao.columns.name = None
    dfs_processados.append(df_secao)
    print(f"[{i}/{len(arquivos_boletim_urna)}] {nome_arq} processado ({len(df_secao):,} seções).")

# 3. Concatenação nacional
df_bu_final = pd.concat(dfs_processados, ignore_index=True)

# 4. Ajuste dos tipos numéricos de seção
colunas_metricas = ['QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES']
for col in colunas_metricas:
    df_bu_final[col] = pd.to_numeric(df_bu_final[col], errors='coerce').fillna(0).astype(int)


Iniciando consolidação de 28 arquivos de Boletim de Urna...
[1/28] bweb_2t_AC_311020221535.csv processado (2,124 seções).
[2/28] bweb_2t_AL_311020221535.csv processado (6,626 seções).
[3/28] bweb_2t_AM_311020221535.csv processado (7,453 seções).
[4/28] bweb_2t_AP_311020221535.csv processado (1,740 seções).
[5/28] bweb_2t_BA_311020221535.csv processado (34,424 seções).
[6/28] bweb_2t_CE_311020221535.csv processado (22,796 seções).
[7/28] bweb_2t_DF_311020221535.csv processado (6,748 seções).
[8/28] bweb_2t_ES_311020221535.csv processado (9,239 seções).
[9/28] bweb_2t_GO_311020221535.csv processado (14,620 seções).
[10/28] bweb_2t_MA_311020221535.csv processado (16,423 seções).
[11/28] bweb_2t_MG_311020221535.csv processado (49,981 seções).
[12/28] bweb_2t_MS_311020221535.csv processado (6,912 seções).
[13/28] bweb_2t_MT_311020221535.csv processado (7,652 seções).
[14/28] bweb_2t_PA_311020221535.csv processado (18,235 seções).
[15/28] bweb_2t_PB_311020221535.csv processado (9,602 seções)

In [13]:
df_bu_final.tail()

,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_URNA_EFETIVADA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCOES,Branco,JAIR BOLSONARO,LULA,Nulo
472023,ZZ,99180,NASSAU,1,1228,1255254,65,25,40,0.0,14.0,9.0,2.0
472024,ZZ,99287,LUSACA,1,1259,1273414,34,14,20,0.0,1.0,13.0,0.0
472025,ZZ,99317,TALIN,1,1282,1042892,250,191,59,1.0,23.0,156.0,11.0
472026,ZZ,99430,KINGSTON-JAMAICA,1,145,1013789,107,21,86,0.0,10.0,11.0,0.0
472027,ZZ,99473,BAREIN,1,1327,1273426,50,40,10,0.0,25.0,11.0,4.0


In [14]:
df_bu_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472028 entries, 0 to 472027
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   SG_UF              472028 non-null  object 
 1   CD_MUNICIPIO       472028 non-null  object 
 2   NM_MUNICIPIO       472028 non-null  object 
 3   NR_ZONA            472028 non-null  object 
 4   NR_SECAO           472028 non-null  object 
 5   NR_URNA_EFETIVADA  472028 non-null  object 
 6   QT_APTOS           472028 non-null  int64  
 7   QT_COMPARECIMENTO  472028 non-null  int64  
 8   QT_ABSTENCOES      472028 non-null  int64  
 9   Branco             472028 non-null  float64
 10  JAIR BOLSONARO     472028 non-null  float64
 11  LULA               472028 non-null  float64
 12  Nulo               472028 non-null  float64
dtypes: float64(4), int64(3), object(6)
memory usage: 46.8+ MB


### 2) Criar dataframe de mapeamento modelo urna e numero

In [5]:
df_modelos_urnas = pd.read_csv(
    "modelourna_numerointerno/modelourna_numerointerno.csv",
    encoding='latin1',
    sep=";"
)

In [6]:
df_modelos_urnas.head()

,DS_MODELO_URNA,NR_FAIXA_INICIAL,NR_FAIXA_FINAL
0,2009,999500,1220500
1,2010,1220501,1345500
2,2011,1368501,1370500
3,2011,1600000,1650000
4,2013,1650001,1701000


In [8]:
df_modelos_urnas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   DS_MODELO_URNA    7 non-null      int64
 1   NR_FAIXA_INICIAL  7 non-null      int64
 2   NR_FAIXA_FINAL    7 non-null      int64
dtypes: int64(3)
memory usage: 296.0 bytes


### função de mapeamento do intervalo e o modelo da urna
O numero da urna efetivada é oque definirá o modelo usado o df de modelos de urnas contem o range valores de numeros de urna para cada modelo vamos criar uma funcao de "de - para" identificando os ranges de cada numero

In [9]:
def categorizar_modelo_urna(series_urna, df_regras):
    urna = pd.to_numeric(series_urna, errors='coerce')
    
    condicoes = [
        urna.between(row['NR_FAIXA_INICIAL'], row['NR_FAIXA_FINAL'])
        for _, row in df_regras.iterrows()
    ]
    modelos = df_regras['DS_MODELO_URNA'].astype(str).tolist()
    
    return np.select(condicoes, modelos, default='OUTRO')

### criando coluna com o modelo da urna no dataframe de boletim

In [16]:
# 5. Classificação dos modelos de urna
df_bu_final['DS_MODELO_URNA'] = categorizar_modelo_urna(df_bu_final['NR_URNA_EFETIVADA'], df_modelos_urnas)
df_bu_final['IS_UE2020'] = (df_bu_final['DS_MODELO_URNA'] == '2020').astype(int)

print("\n--- Consolidação Concluída ---")
print(f"Total de seções no Brasil: {len(df_bu_final):,}")
print(df_bu_final['DS_MODELO_URNA'].value_counts())


--- Consolidação Concluída ---
Total de seções no Brasil: 472,028
DS_MODELO_URNA
2020     192691
2010      93795
2015      85735
2009      48245
2011      26889
2013      24672
OUTRO         1
Name: count, dtype: int64


In [18]:
df_bu_final[df_bu_final['DS_MODELO_URNA'] == 'OUTRO']

,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_URNA_EFETIVADA,QT_APTOS,QT_COMPARECIMENTO,QT_ABSTENCOES,Branco,JAIR BOLSONARO,LULA,Nulo,DS_MODELO_URNA,IS_UE2020
471188,ZZ,29564,CIUDAD GUAYANA,1,91,-1,75,0,75,0.0,0.0,0.0,0.0,OUTRO,0


### Vamos remover essa unica linha com modelo OUTRO o valor n estava no banco de orgiem segundo a documentação

In [19]:
df_bu_final = df_bu_final[df_bu_final['DS_MODELO_URNA'] != 'OUTRO'].copy()

In [20]:
df_bu_final['DS_MODELO_URNA'].value_counts()

DS_MODELO_URNA
2020    192691
2010     93795
2015     85735
2009     48245
2011     26889
2013     24672
Name: count, dtype: int64

# 3) Unir df_bu_final com os dados de perfil eleitorado
O papel do Perfil do Eleitorado

A base consolidada de resultados (df_bu_final) já traz a votação por seção e o modelo de urna utilizado. O objetivo do Perfil do Eleitorado é fornecer as variáveis de confusão demográficas para responder se a variação nos votos persiste após isolar o perfil dos votantes de cada seção:

Faixa etária: Proporção de eleitores jovens vs. idosos por seção.

Escolaridade: Proporção de analfabetos/fundamental incompleto vs. ensino superior.

Gênero: Proporção de eleitores do sexo feminino/masculino.

## estratégia de agregação para unir em dataframe final
1. Diagnóstico dos Dados do PerfilMétrica numérica: Toda agregação deve somar a coluna QT_ELEITORES_PERFIL.
- Gênero (DS_GENERO): MASCULINO e FEMININO.
- Escolaridade (CD_GRAU_ESCOLARIDADE / DS_GRAU_ESCOLARIDADE):Baixa escolaridade: 1 (Analfabeto) e 2 (Lê e escreve).Fundamental: 3 (Incompleto) e 4 (Completo).Médio: 5 (Incompleto) e 6 (Completo).Superior: 7 (Incompleto) e 8 (Superior completo).
- Faixa Etária (CD_FAIXA_ETARIA): Códigos numéricos ordenados onde é simples agrupar jovens (ex.: $\le 24$ anos) e idosos ($\ge 60$ anos).
2. Estratégia de Agregação por Seção Para cruzar com o df_bu_final, precisamos transformar essas centenas de linhas por seção em 1 única linha com percentuais:
- PROP_FEMININO: % de eleitores mulheres na seção.
- PROP_SUPERIOR: % de eleitores com ensino superior (completo ou incompleto).
- PROP_BAIXA_ESCOLARIDADE: % de analfabetos ou apenas lê/escreve.
- PROP_JOVENS: % de eleitores de 16 a 24 anos.
- PROP_IDOSOS: % de eleitores com 60 anos ou mais.

In [22]:
pasta_perfil = "perfil_eleitorado_2022" 
arquivos_perfil = sorted(glob.glob(os.path.join(pasta_perfil, "*.csv")))

colunas_uteis = [
    'SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO',
    'CD_GENERO', 'CD_GRAU_ESCOLARIDADE', 'CD_FAIXA_ETARIA',
    'QT_ELEITORES_PERFIL'
]

chaves_secao = ['SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']
lista_secoes_agregadas = []

print(f"Iniciando agregação de {len(arquivos_perfil)} arquivos de Perfil do Eleitorado...")

for i, arquivo in enumerate(arquivos_perfil, 1):
    nome_arq = os.path.basename(arquivo)
    
    # 1. Lê apenas as colunas necessárias para economizar memória
    df_temp = pd.read_csv(
        arquivo,
        sep=';',
        encoding='latin1',
        usecols=colunas_uteis,
        dtype={'SG_UF': str, 'CD_MUNICIPIO': str, 'NR_ZONA': str, 'NR_SECAO': str},
        on_bad_lines='skip'
    )
    
    # 2. Conversões numéricas vetorizadas
    df_temp['QT_ELEITORES_PERFIL'] = pd.to_numeric(df_temp['QT_ELEITORES_PERFIL'], errors='coerce').fillna(0)
    df_temp['CD_GRAU_ESCOLARIDADE'] = pd.to_numeric(df_temp['CD_GRAU_ESCOLARIDADE'], errors='coerce')
    df_temp['CD_FAIXA_ETARIA'] = pd.to_numeric(df_temp['CD_FAIXA_ETARIA'], errors='coerce')
    df_temp['CD_GENERO'] = pd.to_numeric(df_temp['CD_GENERO'], errors='coerce')

    # 3. Criação de flags quantitativas por categoria
    qtd = df_temp['QT_ELEITORES_PERFIL']
    df_temp['QTD_FEMININO'] = np.where(df_temp['CD_GENERO'] == 4, qtd, 0)
    df_temp['QTD_SUPERIOR'] = np.where(df_temp['CD_GRAU_ESCOLARIDADE'].isin([7, 8]), qtd, 0)
    df_temp['QTD_BAIXA_ESC'] = np.where(df_temp['CD_GRAU_ESCOLARIDADE'].isin([1, 2]), qtd, 0)
    df_temp['QTD_JOVENS'] = np.where(df_temp['CD_FAIXA_ETARIA'] <= 2400, qtd, 0)
    df_temp['QTD_IDOSOS'] = np.where(df_temp['CD_FAIXA_ETARIA'] >= 6000, qtd, 0)

    # 4. Agrega na hora: colapsa centenas de linhas para 1 linha por seção
    df_agg = df_temp.groupby(chaves_secao, as_index=False).agg({
        'QT_ELEITORES_PERFIL': 'sum',
        'QTD_FEMININO': 'sum',
        'QTD_SUPERIOR': 'sum',
        'QTD_BAIXA_ESC': 'sum',
        'QTD_JOVENS': 'sum',
        'QTD_IDOSOS': 'sum'
    })

    # 5. Calcula as proporções sociodemográficas da seção
    total = df_agg['QT_ELEITORES_PERFIL'].replace(0, np.nan)
    df_agg['PROP_FEMININO'] = (df_agg['QTD_FEMININO'] / total).round(4)
    df_agg['PROP_SUPERIOR'] = (df_agg['QTD_SUPERIOR'] / total).round(4)
    df_agg['PROP_BAIXA_ESC'] = (df_agg['QTD_BAIXA_ESC'] / total).round(4)
    df_agg['PROP_JOVENS'] = (df_agg['QTD_JOVENS'] / total).round(4)
    df_agg['PROP_IDOSOS'] = (df_agg['QTD_IDOSOS'] / total).round(4)

    # Mantém apenas a chave e as proporções prontas
    colunas_finais = chaves_secao + [
        'PROP_FEMININO', 'PROP_SUPERIOR', 'PROP_BAIXA_ESC',
        'PROP_JOVENS', 'PROP_IDOSOS', 'QT_ELEITORES_PERFIL'
    ]
    lista_secoes_agregadas.append(df_agg[colunas_finais])
    print(f"[{i}/{len(arquivos_perfil)}] {nome_arq} agregado ({len(df_agg):,} seções).")

# 6. Concatena os resumos nacionais já leves
df_perfil_final = pd.concat(lista_secoes_agregadas, ignore_index=True)
print(f"\nAgregação concluída! Total de seções com perfil: {len(df_perfil_final):,}")

Iniciando agregação de 28 arquivos de Perfil do Eleitorado...
[1/28] perfil_eleitor_secao_2022_AC.csv agregado (2,280 seções).
[2/28] perfil_eleitor_secao_2022_AL.csv agregado (6,816 seções).
[3/28] perfil_eleitor_secao_2022_AM.csv agregado (7,938 seções).
[4/28] perfil_eleitor_secao_2022_AP.csv agregado (1,833 seções).
[5/28] perfil_eleitor_secao_2022_BA.csv agregado (36,837 seções).
[6/28] perfil_eleitor_secao_2022_CE.csv agregado (24,826 seções).
[7/28] perfil_eleitor_secao_2022_DF.csv agregado (6,975 seções).
[8/28] perfil_eleitor_secao_2022_ES.csv agregado (9,605 seções).
[9/28] perfil_eleitor_secao_2022_GO.csv agregado (15,399 seções).
[10/28] perfil_eleitor_secao_2022_MA.csv agregado (19,485 seções).
[11/28] perfil_eleitor_secao_2022_MG.csv agregado (52,474 seções).
[12/28] perfil_eleitor_secao_2022_MS.csv agregado (7,137 seções).
[13/28] perfil_eleitor_secao_2022_MT.csv agregado (8,453 seções).
[14/28] perfil_eleitor_secao_2022_PA.csv agregado (19,704 seções).
[15/28] perfil_el

In [23]:
df_perfil_final.head()

,SG_UF,CD_MUNICIPIO,NR_ZONA,NR_SECAO,PROP_FEMININO,PROP_SUPERIOR,PROP_BAIXA_ESC,PROP_JOVENS,PROP_IDOSOS,QT_ELEITORES_PERFIL
0,AC,1007,9,1,0.4597,0.1313,0.3224,0.1164,0.2567,335
1,AC,1007,9,10,0.4838,0.0361,0.2960,0.1516,0.1227,277
2,AC,1007,9,120,0.4826,0.1262,0.2303,0.0883,0.1483,317
3,AC,1007,9,136,0.4686,0.0221,0.3358,0.1550,0.1439,271
4,AC,1007,9,137,0.4890,0.1041,0.1987,0.1104,0.1640,317


# 4) merge com bu_final

In [24]:
# Garante tipos idênticos nas chaves antes do merge
chaves_merge = ['SG_UF', 'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']
for col in chaves_merge:
    df_bu_final[col] = df_bu_final[col].astype(str).str.strip()
    df_perfil_final[col] = df_perfil_final[col].astype(str).str.strip()

# DataFrame Analítico Completo
df_analise = pd.merge(df_bu_final, df_perfil_final, on=chaves_merge, how='inner')
print(f"Dataset analítico final pronto com {len(df_analise):,} seções integradas!")

Dataset analítico final pronto com 471,691 seções integradas!


In [25]:
# 1. Total de votos válidos e proporção de Lula vs Bolsonaro na base
votos_lula = df_analise['LULA'].sum()
votos_bolsonaro = df_analise['JAIR BOLSONARO'].sum()
total_validos = votos_lula + votos_bolsonaro

print(f"Lula: {votos_lula:,} ({votos_lula / total_validos * 100:.2f}%)")
print(f"Bolsonaro: {votos_bolsonaro:,} ({votos_bolsonaro / total_validos * 100:.2f}%)")

# 2. Distribuição da variável de tratamento (UE2020 vs Legadas)
print("\nDistribuição das urnas integradas:")
print(df_analise['DS_MODELO_URNA'].value_counts(dropna=False))
print(df_analise['IS_UE2020'].value_counts(normalize=True).rename({0: 'Legadas', 1: 'UE2020'}))

# 3. Verificação de valores nulos nas variáveis sociodemográficas
colunas_controle = ['PROP_FEMININO', 'PROP_SUPERIOR', 'PROP_BAIXA_ESC', 'PROP_JOVENS', 'PROP_IDOSOS']
print("\nValores nulos nas variáveis de controle:")
print(df_analise[colunas_controle].isna().sum())

Lula: 60,320,690.0 (50.90%)
Bolsonaro: 58,188,887.0 (49.10%)

Distribuição das urnas integradas:
DS_MODELO_URNA
2020    192507
2010     93772
2015     85657
2009     48214
2011     26877
2013     24664
Name: count, dtype: int64
IS_UE2020
Legadas    0.591879
UE2020     0.408121
Name: proportion, dtype: float64

Valores nulos nas variáveis de controle:
PROP_FEMININO     0
PROP_SUPERIOR     0
PROP_BAIXA_ESC    0
PROP_JOVENS       0
PROP_IDOSOS       0
dtype: int64


# salvando df final de analise como parquet para persistencia

In [30]:
df_analise.to_csv("df_analise_eleicoes_2022.csv.gz", sep=";", index=False, compression='gzip')
print("Backup temporário salvo!")

Backup temporário salvo!


In [ ]:
pip install pyarrow

In [ ]:
pip install fastparquet

In [32]:
df_analise.to_parquet("df_analise_eleicoes_2022.parquet", engine='fastparquet', compression='snappy', index=False)